# Alpha Research Platform

A systematic equity research engine. It builds a set of alpha signals, scores them by
Information Coefficient, blends them by conviction, runs the blend through a risk model and
a constrained optimizer, and backtests the result walk-forward with realistic transaction
costs.

The notebook runs top to bottom with no configuration. API keys are optional — add them
through Colab Secrets to pull live data instead of the built-in fallback.

## Setup

In [ ]:
!pip install -q yfinance fredapi statsmodels scikit-learn plotly 2>/dev/null

import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import spearmanr
from dataclasses import dataclass

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    HAS_PLOTLY = True
except Exception:
    HAS_PLOTLY = False

plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams.update({"figure.figsize": (12, 5), "axes.titleweight": "bold",
                     "axes.titlesize": 12, "font.size": 10})
PALETTE = ["#2C6E91", "#E8743B", "#19A979", "#945ECF", "#C8133B", "#6F7072"]

CONFIG = {
    "tickers": ["AAPL","MSFT","NVDA","AMZN","GOOGL","META","JPM","BAC","XOM","CVX",
                "JNJ","PFE","PG","KO","WMT","HD","UNH","V","MA","DIS",
                "INTC","CSCO","ORCL","ADBE","CRM","NKE","MCD","COST","T","VZ",
                "BA","CAT","GE","MMM","HON","LMT","GS","MS","C","WFC"],
    "start": "2015-01-01", "end": "2024-12-31",
    "rebalance": "M", "cost_bps": 10.0, "cov_lookback": 252,
    "risk_aversion": 8.0, "max_weight": 0.08, "turnover_penalty": 0.001, "min_ic": 0.0,
}

## API Keys

Optional. Leave blank to run on free data, or store keys in Colab Secrets (the key icon in
the left sidebar) using these exact names — they'll load automatically and stay out of the
file.

Fundamentals are sourced FMP → SEC EDGAR → price proxy, in that order. The risk-free rate
comes from FRED if a key is present, otherwise 0%.

In [ ]:
API_KEYS = {
    "SEC_USER_AGENT": "",          # "Name email@example.com" (no signup needed)
    "FRED_API_KEY": "",            # fred.stlouisfed.org/docs/api/api_key.html
    "FMP_API_KEY": "",             # site.financialmodelingprep.com/developer/docs
    "ALPHA_VANTAGE_API_KEY": "",   # alphavantage.co/support/#api-key
    "POLYGON_API_KEY": "",         # polygon.io/dashboard/signup
}

try:
    from google.colab import userdata
    for k in API_KEYS:
        if not API_KEYS[k]:
            try:
                API_KEYS[k] = userdata.get(k) or ""
            except Exception:
                pass
except Exception:
    pass

active = [k for k, v in API_KEYS.items() if v]
print("Active credentials:", active or "none (using free fallbacks)")

## Data

Prices come from Yahoo Finance. If the download fails (offline, rate limited), a synthetic
market with realistic factor structure is generated so the pipeline still runs.

In [ ]:
def fetch_live_prices(tickers, start, end):
    try:
        import yfinance as yf
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
        px = raw["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw
        px = px.dropna(how="all").ffill().dropna(how="all", axis=1)
        if px.shape[1] < 10 or len(px) < 252:
            return None
        print(f"Loaded live prices: {px.shape}")
        return px
    except Exception:
        return None


def generate_synthetic_market(n_assets=40, n_days=2600, seed=42):
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2015-01-01", periods=n_days)
    tickers = CONFIG["tickers"][:n_assets]

    mkt = rng.normal(0.0003, 0.011, n_days)
    sector_id = rng.integers(0, 5, n_assets)
    sector_rets = rng.normal(0.0001, 0.006, (n_days, 5))
    betas = rng.uniform(0.6, 1.4, n_assets)
    quality = rng.normal(0, 1, n_assets)
    value = rng.normal(0, 1, n_assets)

    rets = np.zeros((n_days, n_assets))
    for j in range(n_assets):
        idio = rng.normal(0, 0.013, n_days)
        drift = 0.00010 * quality[j] + 0.00008 * value[j]
        rets[:, j] = betas[j]*mkt + sector_rets[:, sector_id[j]] + idio + drift

    px = pd.DataFrame(100*np.exp(np.cumsum(rets, axis=0)), index=dates, columns=tickers)
    fund = pd.DataFrame({
        "book_to_price": np.exp(0.4*value + rng.normal(0, 0.2, n_assets)),
        "earnings_yield": 0.05 + 0.02*value + rng.normal(0, 0.01, n_assets),
        "roe": 0.10 + 0.05*quality + rng.normal(0, 0.02, n_assets),
    }, index=tickers)
    print(f"Generated synthetic market: {px.shape}")
    return px, fund


def proxy_fundamentals(px):
    long_ret = px.iloc[-1] / px.iloc[0] - 1
    vol = px.pct_change().std()
    return pd.DataFrame({
        "book_to_price": (-long_ret).rank() / len(long_ret) + 0.5,
        "earnings_yield": (-long_ret).rank() / len(long_ret) * 0.04 + 0.03,
        "roe": (-vol).rank() / len(vol) * 0.1 + 0.05,
    }, index=px.columns)


def fetch_fred_riskfree(api_key, series="DGS3MO"):
    if not api_key:
        return 0.0
    try:
        from fredapi import Fred
        rf = float(Fred(api_key=api_key).get_series(series).dropna().iloc[-1]) / 100.0
        print(f"FRED risk-free rate: {rf:.2%}")
        return rf
    except Exception:
        return 0.0


def fetch_fmp_fundamentals(tickers, api_key):
    if not api_key:
        return None
    import requests
    rows = {}
    try:
        for t in tickers:
            url = f"https://financialmodelingprep.com/api/v3/ratios-ttm/{t}?apikey={api_key}"
            r = requests.get(url, timeout=12).json()
            d = r[0] if isinstance(r, list) and r else {}
            pb = d.get("priceToBookRatioTTM") or d.get("priceBookValueRatioTTM")
            rows[t] = {"book_to_price": (1/pb) if pb else np.nan,
                       "earnings_yield": d.get("earningsYieldTTM", np.nan),
                       "roe": d.get("returnOnEquityTTM", np.nan)}
        df = pd.DataFrame(rows).T
        if df.dropna(how="all").empty:
            return None
        print(f"FMP fundamentals: {df.notna().any(axis=1).sum()} names")
        return df
    except Exception:
        return None


def fetch_edgar_roe(tickers, user_agent):
    if not user_agent:
        return None
    import requests
    h = {"User-Agent": user_agent}
    try:
        cik_map = requests.get("https://www.sec.gov/files/company_tickers.json",
                               headers=h, timeout=20).json()
        t2cik = {v["ticker"]: str(v["cik_str"]).zfill(10) for v in cik_map.values()}

        def latest(facts, tag):
            try:
                u = facts["us-gaap"][tag]["units"]["USD"]
                annual = [x for x in u if x.get("form", "").startswith("10-K")]
                return sorted(annual or u, key=lambda x: x.get("end", ""))[-1]["val"]
            except Exception:
                return np.nan

        rows = {}
        for t in tickers:
            cik = t2cik.get(t)
            if not cik:
                continue
            facts = requests.get(
                f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json",
                headers=h, timeout=20).json().get("facts", {})
            eq, ni = latest(facts, "StockholdersEquity"), latest(facts, "NetIncomeLoss")
            if eq and np.isfinite(eq) and eq != 0 and np.isfinite(ni):
                rows[t] = {"roe": ni / eq}
        if not rows:
            return None
        print(f"EDGAR ROE: {len(rows)} names")
        return pd.DataFrame(rows).T
    except Exception:
        return None


def assemble_fundamentals(px, keys):
    fund = proxy_fundamentals(px)
    fmp = fetch_fmp_fundamentals(list(px.columns), keys.get("FMP_API_KEY", ""))
    if fmp is not None:
        for c in ["book_to_price", "earnings_yield", "roe"]:
            if c in fmp:
                fund[c] = fmp[c].reindex(fund.index).fillna(fund[c])
    edgar = fetch_edgar_roe(list(px.columns), keys.get("SEC_USER_AGENT", ""))
    if edgar is not None:
        fund["roe"] = edgar["roe"].reindex(fund.index).fillna(fund["roe"])
    for c in fund.columns:
        fund[c] = fund[c].fillna(fund[c].median())
    return fund


RF_ANNUAL = fetch_fred_riskfree(API_KEYS["FRED_API_KEY"])
prices = fetch_live_prices(CONFIG["tickers"], CONFIG["start"], CONFIG["end"])
if prices is None:
    prices, fundamentals = generate_synthetic_market()
    source = "synthetic"
else:
    fundamentals = assemble_fundamentals(prices, API_KEYS)
    source = "Yahoo Finance"

print(f"\nSource: {source} | {prices.shape[1]} assets | "
      f"{prices.index.min().date()} to {prices.index.max().date()}")

## Cleaning

Forward-fill gaps and keep fully populated names. The forward-return matrix is shifted by
one day so a signal known on day *t* is only ever matched to the return from *t* to *t+1*.

In [ ]:
prices = prices.ffill().dropna(axis=1)
fundamentals = fundamentals.reindex(prices.columns).dropna()
prices = prices[fundamentals.index]
returns = prices.pct_change()
fwd_return = returns.shift(-1)
print(f"Universe: {prices.shape[1]} assets, {len(prices)} days")

## Alpha Library

Each signal is standardized cross-sectionally (winsorized z-score per day), so a higher
score means more attractive and signals are comparable for blending.

In [ ]:
def winsor_z(df, lower=0.02, upper=0.98):
    def row(r):
        r = r.clip(r.quantile(lower), r.quantile(upper))
        s = r.std(ddof=0)
        return (r - r.mean()) / s if s > 0 else r * 0.0
    return df.apply(row, axis=1)


class AlphaLibrary:
    def __init__(self, prices, fundamentals):
        self.px = prices
        self.rets = prices.pct_change()
        self.fund = fundamentals

    def momentum(self, lookback=252, skip=21):
        return winsor_z(self.px.shift(skip)/self.px.shift(lookback) - 1).dropna(how="all")

    def short_reversal(self, lookback=5):
        return winsor_z(-(self.px/self.px.shift(lookback) - 1)).dropna(how="all")

    def low_volatility(self, lookback=126):
        return winsor_z(-self.rets.rolling(lookback).std()).dropna(how="all")

    def _broadcast(self, s):
        return pd.DataFrame(np.tile(s.values, (len(self.px), 1)),
                            index=self.px.index, columns=self.px.columns)

    def value(self):
        s = (winsor_z(self.fund[["book_to_price"]].T).iloc[0]
             + winsor_z(self.fund[["earnings_yield"]].T).iloc[0]) / 2
        return self._broadcast(s)

    def quality(self):
        return self._broadcast(winsor_z(self.fund[["roe"]].T).iloc[0])

    def all_alphas(self):
        return {"momentum": self.momentum(), "short_reversal": self.short_reversal(),
                "low_vol": self.low_volatility(), "value": self.value(),
                "quality": self.quality()}


alphas = AlphaLibrary(prices, fundamentals).all_alphas()
print("Alphas:", list(alphas.keys()))

## Signal Evaluation

The Information Coefficient is the daily rank correlation between a signal and the forward
return. Its mean measures predictive power; ICIR (mean / std) measures consistency.

In [ ]:
def information_coefficient(alpha, fwd):
    idx = alpha.index.intersection(fwd.index)
    ics = []
    for dt in idx:
        a, f = alpha.loc[dt], fwd.loc[dt]
        m = a.notna() & f.notna()
        ics.append(spearmanr(a[m], f[m])[0] if m.sum() >= 5 else np.nan)
    s = pd.Series(ics, index=idx).dropna()
    return s, {"mean_IC": s.mean(), "ICIR": s.mean()/s.std() if s.std() > 0 else np.nan,
               "hit_rate": (s > 0).mean()}


ic_summaries = {n: information_coefficient(a, fwd_return)[1] for n, a in alphas.items()}
ic_table = pd.DataFrame(ic_summaries).T
display(ic_table.style.format("{:.4f}").background_gradient(
    subset=["mean_IC", "ICIR"], cmap="RdYlGn"))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
vals = ic_table["mean_IC"].sort_values()
ax.barh(vals.index, vals.values,
        color=[PALETTE[2] if v > 0 else PALETTE[4] for v in vals],
        edgecolor="black", lw=0.6)
for i, (name, v) in enumerate(vals.items()):
    ax.text(v, i, f"  ICIR {ic_table.loc[name,'ICIR']:.2f}",
            va="center", ha="left" if v >= 0 else "right", fontsize=9)
ax.axvline(0, color="black", lw=1)
ax.set_title("Mean Information Coefficient by Signal")
ax.set_xlabel("Mean rank-IC")
plt.tight_layout(); plt.show()

## Combination

Signals are weighted by ICIR; anything with non-positive mean IC is dropped. The blend is
re-standardized.

In [ ]:
def combine_alphas(alphas, ic_summaries, min_ic=0.0):
    w = {n: max(ic_summaries[n]["ICIR"], 0.0) for n, a in alphas.items()
         if ic_summaries[n]["mean_IC"] > min_ic and np.isfinite(ic_summaries[n]["ICIR"])}
    if not w:
        raise ValueError("No alphas passed the IC screen.")
    tot = sum(w.values())
    w = {k: v/tot for k, v in w.items()}
    cols = list(alphas.values())[0].columns
    combined = None
    for n, wt in w.items():
        f = alphas[n].reindex(columns=cols)
        combined = f*wt if combined is None else combined.add(f*wt, fill_value=0)
    return winsor_z(combined).dropna(how="all"), w


combined_alpha, blend_weights = combine_alphas(alphas, ic_summaries, CONFIG["min_ic"])
for k, v in sorted(blend_weights.items(), key=lambda x: -x[1]):
    print(f"{k:14s} {v:.1%}")

## Risk Model

A raw sample covariance is noisy and nearly singular when assets are close in number to
observations. Shrinking toward a constant-correlation target stabilizes the optimizer.

In [ ]:
def shrink_covariance(returns, shrink=0.3):
    R = returns.dropna(how="any")
    S = R.cov().values
    n = S.shape[0]
    std = np.sqrt(np.diag(S))
    corr = S / np.outer(std, std)
    avg = (corr.sum() - n) / (n*(n-1))
    target = avg * np.outer(std, std)
    np.fill_diagonal(target, np.diag(S))
    Sigma = shrink*target + (1-shrink)*S
    return pd.DataFrame(Sigma, index=R.columns, columns=R.columns)


Sigma = shrink_covariance(returns)
print(f"Condition number: {np.linalg.cond(Sigma.values):,.0f} "
      f"(raw: {np.linalg.cond(returns.dropna().cov().values):,.0f})")

## Optimizer

Maximize `alpha − ½·λ·variance − tc·turnover`, long-only, fully invested, with a per-name
cap. Solved with SLSQP, so no external solver is required.

In [ ]:
def optimize_weights(alpha_scores, Sigma, risk_aversion=8.0, max_weight=0.08,
                     prev_w=None, turnover_penalty=0.0):
    assets = Sigma.columns
    a = alpha_scores.reindex(assets).fillna(0).values
    S = Sigma.values
    n = len(assets)
    w0 = np.full(n, 1/n) if prev_w is None else prev_w.reindex(assets).fillna(0).values

    def neg_obj(w):
        u = a @ w - 0.5*risk_aversion*(w @ S @ w)
        if turnover_penalty > 0:
            u -= turnover_penalty * np.abs(w - w0).sum()
        return -u

    res = minimize(neg_obj, w0, method="SLSQP", bounds=[(0, max_weight)]*n,
                   constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
                   options={"maxiter": 200, "ftol": 1e-9})
    w = np.clip(res.x, 0, max_weight)
    w = w/w.sum() if w.sum() > 0 else np.full(n, 1/n)
    return pd.Series(w, index=assets)

## Backtest

Monthly rebalance, walk-forward. On each date the covariance uses a trailing window and the
alpha is the latest available reading — both strictly past-looking. Turnover is charged at
the configured cost.

In [ ]:
def safe_freq(alias):
    try:
        pd.tseries.frequencies.to_offset(alias); return alias
    except Exception:
        return {"M": "ME", "Q": "QE", "Y": "YE"}.get(alias, alias)


@dataclass
class BacktestResult:
    equity: pd.Series; weights: pd.DataFrame; turnover: pd.Series
    gross_returns: pd.Series; net_returns: pd.Series


def run_backtest(prices, combined_alpha, rebalance="M", cost_bps=10.0,
                 risk_aversion=8.0, max_weight=0.08, cov_lookback=252,
                 turnover_penalty=0.001):
    rets = prices.pct_change().fillna(0)
    dates = [d for d in prices.resample(safe_freq(rebalance)).last().index
             if d in prices.index]
    weights_hist, prev_w = {}, None
    net_chunks, gross_chunks, turn = [], [], {}

    for i, dt in enumerate(dates[:-1]):
        window = prices.loc[:dt].tail(cov_lookback)
        if len(window) < 60:
            continue
        Sigma = shrink_covariance(window.pct_change())
        valid = Sigma.columns
        past_alpha = combined_alpha.loc[:dt]
        if past_alpha.empty:
            continue
        a = past_alpha.iloc[-1].reindex(valid)
        w = optimize_weights(a, Sigma, risk_aversion, max_weight, prev_w, turnover_penalty)
        weights_hist[dt] = w

        to = 1.0 if prev_w is None else (
            w.reindex(valid).fillna(0) - prev_w.reindex(valid).fillna(0)).abs().sum()/2
        turn[dt] = to

        hold = rets.loc[dt:dates[i+1]].iloc[1:]
        port = (hold[valid] * w.reindex(valid).fillna(0)).sum(axis=1)
        if len(port):
            net = port.copy()
            net.iloc[0] -= to * cost_bps / 1e4
            net_chunks.append(net); gross_chunks.append(port)
        prev_w = w

    gross = pd.concat(gross_chunks).sort_index()
    net = pd.concat(net_chunks).sort_index()
    return BacktestResult((1+net).cumprod(), pd.DataFrame(weights_hist).T.sort_index(),
                          pd.Series(turn).sort_index(), gross, net)


result = run_backtest(prices, combined_alpha, CONFIG["rebalance"], CONFIG["cost_bps"],
                      CONFIG["risk_aversion"], CONFIG["max_weight"],
                      CONFIG["cov_lookback"], CONFIG["turnover_penalty"])
print(f"{len(result.weights)} rebalances | final equity {result.equity.iloc[-1]:.2f}x")

## Performance

In [ ]:
def performance_metrics(r, freq=252, rf=0.0):
    r = r.dropna()
    yrs = len(r)/freq
    cagr = (1+r).prod()**(1/yrs) - 1 if yrs > 0 else np.nan
    vol = r.std()*np.sqrt(freq)
    dn = r[r < 0].std()*np.sqrt(freq)
    eq = (1+r).cumprod()
    dd = (eq/eq.cummax() - 1).min()
    return {"CAGR": cagr, "Volatility": vol,
            "Sharpe": (r.mean()*freq - rf)/vol if vol > 0 else np.nan,
            "Sortino": (r.mean()*freq - rf)/dn if dn > 0 else np.nan,
            "Max Drawdown": dd, "Calmar": cagr/abs(dd) if dd < 0 else np.nan,
            "Hit Rate": (r > 0).mean()}


common = result.net_returns.index.intersection(returns.index)
bench = returns.loc[common].mean(axis=1).reindex(result.net_returns.index).fillna(0)
metrics = pd.DataFrame({"Strategy": performance_metrics(result.net_returns, rf=RF_ANNUAL),
                        "Equal-Weight": performance_metrics(bench, rf=RF_ANNUAL)})
display(metrics.style.format("{:.3f}").background_gradient(
    axis=1, cmap="RdYlGn",
    subset=pd.IndexSlice[["CAGR","Sharpe","Sortino","Calmar"], :]))

## Tearsheet

In [ ]:
def plot_tearsheet(result, bench, blend_weights):
    r, eq = result.net_returns, result.equity
    bench_eq = (1+bench).cumprod()
    fig = plt.figure(figsize=(15, 10))
    gs = fig.add_gridspec(3, 2, hspace=0.35, wspace=0.22)

    ax = fig.add_subplot(gs[0, :])
    ax.plot(eq.index, eq.values, color=PALETTE[0], lw=2, label="Strategy")
    ax.plot(bench_eq.index, bench_eq.values, color=PALETTE[5], lw=1.5, ls="--",
            label="Equal-Weight")
    ax.set_title("Cumulative Performance (net of costs)"); ax.legend(loc="upper left")

    ax = fig.add_subplot(gs[1, 0])
    dd = eq/eq.cummax() - 1
    ax.fill_between(dd.index, dd.values, 0, color=PALETTE[4], alpha=0.5)
    ax.set_title("Drawdown")

    ax = fig.add_subplot(gs[1, 1])
    roll = (r.rolling(252).mean()/r.rolling(252).std())*np.sqrt(252)
    ax.plot(roll.index, roll.values, color=PALETTE[3], lw=1.5)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title("Rolling 1-Year Sharpe")

    ax = fig.add_subplot(gs[2, 0])
    mret = r.add(1).resample(safe_freq("M")).prod() - 1
    tbl = mret.groupby([mret.index.year, mret.index.month]).first().unstack()
    lim = abs(tbl.values).max()
    im = ax.imshow(tbl.values, aspect="auto", cmap="RdYlGn", vmin=-lim, vmax=lim)
    ax.set_yticks(range(len(tbl.index))); ax.set_yticklabels(tbl.index)
    ax.set_xticks(range(12)); ax.set_xticklabels(list("JFMAMJJASOND"))
    ax.set_title("Monthly Returns")
    plt.colorbar(im, ax=ax, fraction=0.046)

    ax = fig.add_subplot(gs[2, 1])
    bw = pd.Series(blend_weights).sort_values()
    ax.barh(bw.index, bw.values, color=PALETTE[2], edgecolor="black", lw=0.6)
    ax.set_title(f"Blend Weights (avg turnover {result.turnover.mean():.1%})")

    fig.suptitle("Strategy Tearsheet", fontsize=15, fontweight="bold", y=0.995)
    plt.show()


plot_tearsheet(result, bench, blend_weights)

## Attribution

Regress strategy returns on each signal's long-short return to see what drove the P&L.

In [ ]:
def sleeve_return(alpha, fwd, q=0.2):
    out = {}
    for dt in alpha.index.intersection(fwd.index):
        a, f = alpha.loc[dt], fwd.loc[dt]
        m = a.notna() & f.notna()
        if m.sum() < 10:
            continue
        a, f = a[m], f[m]
        hi, lo = a >= a.quantile(1-q), a <= a.quantile(q)
        if hi.sum() and lo.sum():
            out[dt] = f[hi].mean() - f[lo].mean()
    return pd.Series(out)


import statsmodels.api as sm
sleeves = pd.DataFrame({n: sleeve_return(a, fwd_return) for n, a in alphas.items()})
sleeves = sleeves.reindex(result.net_returns.index).fillna(0)
model = sm.OLS(result.net_returns, sm.add_constant(sleeves)).fit()
attr = pd.DataFrame({"beta": model.params, "t_stat": model.tvalues}).drop("const")
display(attr.style.format("{:.3f}").background_gradient(subset=["beta"], cmap="RdYlGn"))
print(f"R-squared: {model.rsquared:.2f}")

## Dashboard

In [ ]:
if HAS_PLOTLY:
    eq, bench_eq = result.equity, (1+bench).cumprod()
    dd = eq/eq.cummax() - 1
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3],
                        vertical_spacing=0.06,
                        subplot_titles=("Cumulative Performance", "Drawdown"))
    fig.add_trace(go.Scatter(x=eq.index, y=eq.values, name="Strategy",
                             line=dict(color="#2C6E91", width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=bench_eq.index, y=bench_eq.values, name="Equal-Weight",
                             line=dict(color="#888", width=1.5, dash="dash")), row=1, col=1)
    fig.add_trace(go.Scatter(x=dd.index, y=dd.values, name="Drawdown", fill="tozeroy",
                             line=dict(color="#C8133B")), row=2, col=1)
    fig.update_layout(height=600, template="plotly_white", title="Alpha Research Platform",
                      legend=dict(orientation="h", y=1.08))
    fig.update_yaxes(tickformat=".0%", row=2, col=1)
    fig.show()
else:
    print("Install plotly to view the interactive dashboard.")

## Summary

In [ ]:
sm_ = performance_metrics(result.net_returns, rf=RF_ANNUAL)
bm_ = performance_metrics(bench, rf=RF_ANNUAL)
print(f"Source            {source}")
print(f"Universe          {prices.shape[1]} assets")
print(f"Period            {prices.index.min().date()} to {prices.index.max().date()}")
print(f"Alphas combined   {', '.join(blend_weights)}")
print(f"Sharpe            {sm_['Sharpe']:.2f} (benchmark {bm_['Sharpe']:.2f})")
print(f"CAGR              {sm_['CAGR']:.2%}")
print(f"Max drawdown      {sm_['Max Drawdown']:.2%}")
print(f"Avg turnover      {result.turnover.mean():.1%}")